# 02 · RDD Fundamentals

> **Module 2 · Spark Fundamentals · Lecture 2.2**

In this notebook you will:

- Create RDDs with `sc.parallelize` and apply transformations (`map`, `filter`, `flatMap`, `reduce`)
- Trigger work with actions (`collect`, `take`, `first`, `count`, `reduce`) and understand what they cost
- Compare `reduceByKey` and `groupByKey` and explain the shuffle-cost difference
- Use pair RDD operations, `repartition`/`coalesce`, `persist`/`cache`, and `getStorageLevel()`
- Read lineage with `toDebugString()` and compare `mapPartitions` with `map` and `foreachPartition`

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


## 1 · Creating an RDD

`sc.parallelize` ships a local Python collection to the executors and splits it into
partitions. `numSlices` controls the number of parallel tasks.

- Each RDD is **partitioned**, **closure-based**, **lineage-based**, **lazy**, and
  **fault-tolerant** (by recomputation, not replication).
- The function you pass to `map`/`filter` is **serialized** to the executors. It must be
  picklable, and anything it captures is shipped with every task.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


In [ ]:
sc = spark.sparkContext

numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8], numSlices=2)
print("partitions :", numbers.getNumPartitions())
print("collect    :", numbers.collect())
print("first      :", numbers.first())
print("count      :", numbers.count())

# The partition layout is visible before any action.
for i, part in enumerate(numbers.glom().collect()):
    print(f"  partition {i}: {part}")

## 2 · Transformations vs actions

**Transformations are lazy** — they build the lineage. **Actions are eager** — they
execute the job.

| Transformations (lazy) | Actions (eager) |
|---|---|
| `map`, `filter`, `flatMap`, `mapPartitions` | `collect`, `take`, `first`, `count` |
| `reduceByKey`, `groupByKey`, `join` | `reduce`, `foreach`, `saveAsTextFile` |
| `repartition`, `coalesce`, `distinct` | `takeOrdered`, `countByKey` |

In [ ]:
# Transformations
doubled = numbers.map(lambda x: x * 2)
evens = numbers.filter(lambda x: x % 2 == 0)
flattened = sc.parallelize([[1, 2], [3, 4]]).flatMap(lambda xs: xs)
words = sc.parallelize("the quick brown fox".split()).map(lambda w: (w, 1))

print("map      :", doubled.collect())
print("filter   :", evens.collect())
print("flatMap  :", flattened.collect())

# Actions
print("take(3)  :", numbers.take(3))
print("reduce   :", numbers.reduce(lambda a, b: a + b))
print("countByKey:", dict(words.countByKey()))

## 3 · `reduceByKey` vs `groupByKey`

Both shuffle, but only one combines **before** the shuffle.

```
Data:  ("a", 1), ("b", 2), ("a", 3), ("b", 4), ("a", 5)

groupByKey:
   shuffle ALL (key, value) pairs  ->  ("a",[1,3,5]), ("b",[2,4])
   memory on one executor = the size of the ENTIRE group

reduceByKey:
   combine LOCALLY first (map-side combine)  ->  ("a", 1+3+5), ("b", 2+4)
   memory per partition = O(number of keys), not O(group size)
```

**Prefer `reduceByKey`.** `groupByKey` is only necessary when you genuinely need every
value of a group (e.g. to build a list), or when your operation is not commutative and
associative so a local combine would be incorrect.

In [ ]:
pairs = sc.parallelize(
    [("a", 1), ("b", 2), ("a", 3), ("b", 4), ("a", 5), ("c", 1)], numSlices=2
)

print("reduceByKey :", pairs.reduceByKey(lambda x, y: x + y).collect())
print("groupByKey  :", sorted(pairs.groupByKey().mapValues(sum).collect()))

# pairRDD-specific operations
print("keys        :", pairs.keys().collect())
print("values      :", pairs.values().collect())
print("mapValues   :", pairs.mapValues(lambda v: v * 10).collect())
print("flatMapValues:", pairs.flatMapValues(lambda v: [v, -v]).collect())
print("filter key  :", pairs.filter(lambda kv: kv[0] == "a").collect())

# join on the key
other = sc.parallelize([("a", "alpha"), ("b", "beta")])
print("join        :", sorted(pairs.join(other).collect()))

## 4 · `repartition` vs `coalesce`

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Shuffle | **Yes** — full shuffle into `n` partitions | No — merges existing partitions |
| Distribution | Even (round-robin) or by key (`repartition(col)`) | May be uneven |
| Use when | You need even sizing or key-based layout | You just want fewer output files |

In [ ]:
base = sc.parallelize(range(1, 11), numSlices=4)
print("base partitions :", base.getNumPartitions())

r = base.repartition(3)
print("repartition(3)  :", r.getNumPartitions(), sorted(r.collect()))

c = base.coalesce(2)
print("coalesce(2)     :", c.getNumPartitions(), sorted(c.collect()))

# Partition BY KEY keeps every key in exactly one partition.
# PySpark has no `repartition(n, fn)`; use partitionBy() on a (key, value) RDD.
keyed = sc.parallelize([(k % 3, k) for k in range(10)])
by_key = keyed.partitionBy(3)
print("partitionBy(3)   :", by_key.getNumPartitions())

# Each partition now holds a single key.
# groupByKey() returns a lazy ResultIterable, not a list -> wrap it in list().
for key, values in sorted(by_key.groupByKey().collect()):
    print(f"   key {key} -> {sorted(list(values))}")

## 5 · `persist` / `cache` and storage levels

Caching is **lazy** — the data is materialised only when an action runs. `cache()` is
exactly `persist(StorageLevel.MEMORY_AND_DISK)`.

| PySpark constant | Memory | Disk | Survives executor loss |
|---|---|---|---|
| `MEMORY_ONLY` | yes (serialized) | no | no |
| `MEMORY_AND_DISK` | yes | yes | no (**the default**) |
| `MEMORY_AND_DISK_DESER` | yes (deserialized) | yes | no |
| `DISK_ONLY` | no | yes | no |
| `DISK_ONLY_2` | no | yes (2 copies) | yes |
| `MEMORY_AND_DISK_2` | yes | yes (2 copies) | yes |

**Caution:** The PySpark names differ from Scala's. `StorageLevel.MEMORY_ONLY_SER` does **not**
exist in PySpark, and PySpark's `MEMORY_ONLY` is the *serialized* variant.

In [ ]:
from pyspark import StorageLevel

cached = sc.parallelize(range(1, 100000), numSlices=4)
cached.persist(StorageLevel.MEMORY_AND_DISK)
print("storage level :", cached.getStorageLevel())
print("count         :", cached.count())      # action -> materialises the cache

# Inspect the level with its boolean attributes — there is no .contains() in PySpark.
level = cached.getStorageLevel()
print(f"useMemory     : {level.useMemory}")
print(f"useDisk       : {level.useDisk}")
print(f"deserialized  : {level.deserialized}")
print(f"replication   : {level.replication}")
print("still cached  :", level.useMemory and level.useDisk)

cached.unpersist()
print("after unpersist:", cached.getStorageLevel())

## 6 · Lineage and `toDebugString()`

Every RDD remembers how it was produced. That record is the lineage, and it is the
mechanism of fault tolerance: a lost partition is recomputed by replaying its ancestry.

In [ ]:
lineage = (
    sc.parallelize(range(1, 100), numSlices=4)
      .map(lambda x: x * 2)
      .filter(lambda x: x % 3 == 0)
      .map(lambda x: x + 1)
)
print(lineage.toDebugString())

### `count()` shows lineage reuse

Calling `count()` twice on the same RDD recomputes the lineage — unless the RDD is
persisted. The stage IDs in the Spark UI tell the story: a cached RDD produces one
stage; an uncached one produces two.

In [ ]:
uncached = sc.parallelize(range(1, 200000), numSlices=4).map(lambda x: x * 2)
print("uncached count #1:", uncached.count())
print("uncached count #2:", uncached.count())   # lineage replayed

now_cached = sc.parallelize(range(1, 200000), numSlices=4).map(lambda x: x * 2)
now_cached.persist(StorageLevel.MEMORY_AND_DISK)
print("cached count #1  :", now_cached.count())
print("cached count #2  :", now_cached.count())   # reuses the cached blocks
now_cached.unpersist()

## 7 · `mapPartitions` vs `map`

`map` calls your function **once per record**. `mapPartitions` calls it **once per
partition** with an iterator — which is where batching belongs.

**Caution:** `mapPartitions` functions must **yield**; returning a list makes Spark treat the
whole list as a single output element.

In [ ]:
# map: one Python call per row
mapped = sc.parallelize(range(1, 11)).map(lambda x: x * 2)
print("map         :", mapped.collect())

# mapPartitions: one Python call per partition, batch inside
def batch_double(iterator):
    print("    [mapPartitions invoked]")
    for value in iterator:
        yield value * 2

partitioned = sc.parallelize(range(1, 11), numSlices=2).mapPartitions(batch_double)
print("mapPartitions:", partitioned.collect())

## 8 · The `foreachPartition` idiom

`foreachPartition` runs a function on the executors, once per partition, with **no data
flowing back to the driver**. This is the standard pattern for custom writes (JDBC bulk
loads, Pub/Sub publishing, Arrow/Parquet writers) that the built-in `write` API does not
cover.

In [ ]:
def foreach_partition(iterator):
    total = 0
    for value in iterator:
        total += value
    # In production: write to an external system here, not print to the driver.
    print(f"    [executor] partition sum = {total}")

sc.parallelize(range(1, 101), numSlices=4).foreachPartition(foreach_partition)

## 9 · RDD vs DataFrame

| Dimension | RDD | DataFrame |
|---|---|---|
| API | `pyspark.RDD` | `pyspark.sql.DataFrame` |
| Schema | None — Python objects | Known, enforced at runtime |
| Catalyst optimisation | None | Full (predicate pushdown, column pruning, join reordering) |
| Predicate pushdown | None | Yes |
| Serialisation | Java/Pickle per object | Columnar (Tungsten) |
| Typical speed | Slower (per-row Python overhead) | Faster (generated JVM code) |
| UDF support | Python functions | Python UDFs, `pandas_udf`, native SQL |
| Use when | Custom binary/record processing, explicit partition control, library APIs that only accept RDDs | Almost everything else |

**Note:** **Heuristic:** if you can describe the operation in terms of *rows and columns*, use a
DataFrame. If you can only describe it in terms of *bytes, records, and partitions*, use
an RDD.

In [ ]:
# The same computation in both APIs.
rdd_result = sc.parallelize([1, 2, 3, 4]).map(lambda x: x * 2).filter(lambda x: x > 4).collect()
df_result = (
    spark.createDataFrame([(1,), (2,), (3,), (4,)], "v int")
    .withColumn("doubled", F.col("v") * 2)
    .filter(F.col("doubled") > 4)
    .select("doubled")
    .collect()
)
# An RDD of ints gives you plain ints; a DataFrame gives you Row objects.
print("RDD       :", rdd_result)
print("DataFrame :", [r["doubled"] for r in df_result])

# The anti-pattern: df.rdd.map loses the schema and defeats Catalyst.
# Note the result is plain ints again — the Row schema is gone.
anti = spark.createDataFrame([(1,), (2,)], "v int").rdd.map(lambda r: r["v"] * 2).collect()
print("df.rdd.map:", anti)


## 🔑 Key Takeaways

- RDDs are partitioned, closure-based, lineage-based, lazy, and fault-tolerant by recomputation.
- Transformations are lazy; actions (`collect`, `take`, `first`, `count`, `reduce`) are eager.
- **Prefer `reduceByKey` over `groupByKey`** — map-side combine bounds memory by key count rather than group size.
- `repartition` shuffles and can even out distribution; `coalesce` merges partitions without a shuffle.
- `persist()` is lazy — follow it with an action. `cache()` == `persist(StorageLevel.MEMORY_AND_DISK)`.
- `toDebugString()` exposes the lineage; repeated `count()` on an uncached RDD replays it, on a cached one it does not.
- `mapPartitions` runs once per partition (the place to batch); `foreachPartition` is the executor-side write idiom.
- Use a DataFrame unless you need byte-level control, explicit partitioning, or an RDD-only library API.

---

## 📚 Next steps

- Measure `reduceByKey` vs `groupByKey` on a synthetic RDD with one hot key and watch the memory difference.
- Run `03-dataframes-and-schema.ipynb` to move from the RDD API to the DataFrame API.
